# 06 — Attention Head Analysis

**Objective**: Identify which of BERT's 144 attention heads (12 layers x 12 heads) are critical for each of the 28 emotions.

**Methods**:
1. **Head ablation**: Zero out each head's contribution and measure per-emotion F1 drop
2. **Head importance map**: Heatmap of (head, emotion) importance scores
3. **Specialization analysis**: Which heads are emotion-specific vs. general-purpose
4. **Redundancy analysis**: Identify groups of functionally equivalent heads
5. **Attention pattern visualization**: What top specialized heads attend to

This provides the attention-level complement to notebook 07 (FFN neurons).

In [ ]:
import sys, os

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    !pip install -q transformers datasets scikit-learn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

from src.data import load_goemotions
from src.models import load_bert_classifier

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data and model
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)

MODEL_PATH = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-23emo-final')
NUM_EMOTIONS = len(emotion_names)
model = load_bert_classifier(model_name=MODEL_PATH, num_labels=NUM_EMOTIONS)
model.to(device)
model.eval()

NUM_LAYERS = 12
NUM_HEADS = 12
HEAD_DIM = 64  # 768 / 12

print(f"Model loaded. {NUM_LAYERS} layers x {NUM_HEADS} heads = {NUM_LAYERS * NUM_HEADS} total heads")
print(f"Emotions: {NUM_EMOTIONS}")

In [ ]:
# Prepare evaluation dataloader
# Use a subset for faster ablation (full test set = too slow for 144 ablations)
N_EVAL = 2000  # samples for ablation evaluation
eval_subset = test_ds.select(range(min(N_EVAL, len(test_ds))))
eval_loader = DataLoader(eval_subset, batch_size=64, collate_fn=data_collator, shuffle=False)

print(f"Evaluation subset: {len(eval_subset)} samples")

## 1. Baseline Per-Emotion F1

In [ ]:
@torch.no_grad()
def evaluate_per_emotion(model, loader, device):
    """Compute per-emotion F1 scores."""
    all_preds = []
    all_labels = []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits) > 0.5).float()
        
        all_preds.append(preds)
        all_labels.append(labels)
    
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    per_emotion_f1 = []
    for i in range(all_labels.shape[1]):
        if all_labels[:, i].sum() > 0:
            f1 = f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
        else:
            f1 = np.nan
        per_emotion_f1.append(f1)
    
    macro_f1 = np.nanmean(per_emotion_f1)
    return np.array(per_emotion_f1), macro_f1

In [ ]:
baseline_per_emotion, baseline_macro = evaluate_per_emotion(model, eval_loader, device)
print(f"Baseline macro F1: {baseline_macro:.4f}")
print(f"\nPer-emotion F1:")
for i, name in enumerate(emotion_names):
    print(f"  {name:20s}: {baseline_per_emotion[i]:.4f}")

## 2. Head Ablation Study

For each head (layer L, head H), we install a forward hook on the attention output projection (`bert.encoder.layer[L].attention.output.dense`) that zeros out the 64-dimensional slice corresponding to head H. We then measure per-emotion F1 drop.

In [ ]:
class HeadAblationHook:
    """Hook that zeros out a specific head's contribution in the attention output.
    
    Operates on the input to attention.output.dense (i.e., the concatenated
    head outputs before the output projection). This is the standard approach
    for head ablation (see Voita et al., 2019; Michel et al., 2019).
    """
    def __init__(self, head_idx, head_dim=64):
        self.head_idx = head_idx
        self.head_dim = head_dim
        self.handle = None
    
    def hook_fn(self, module, input, output):
        # input[0] has shape (batch, seq_len, 768)
        # We zero out the slice for head_idx: [head_idx*64 : (head_idx+1)*64]
        # But we need to modify the INPUT to output.dense, not output
        # Actually, we hook on attention.self output and zero the head there
        pass
    
    def install(self, layer_module):
        pass
    
    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


def ablate_head_via_attention_self(model, layer_idx, head_idx, head_dim=64):
    """Install hook on attention.self to zero out a head's output.
    
    Hooks on bert.encoder.layer[L].attention.self and modifies the
    context_layer output to zero the specified head dimension.
    Returns the hook handle for cleanup.
    """
    attn_self = model.bert.encoder.layer[layer_idx].attention.self
    
    def hook_fn(module, input, output):
        # output is a tuple: (context_layer, attention_probs) or just (context_layer,)
        context = output[0]  # (batch, seq_len, 768)
        start = head_idx * head_dim
        end = (head_idx + 1) * head_dim
        # Zero out this head's dimensions
        modified = context.clone()
        modified[:, :, start:end] = 0.0
        # Return modified output tuple
        return (modified,) + output[1:]
    
    handle = attn_self.register_forward_hook(hook_fn)
    return handle

In [ ]:
@torch.no_grad()
def run_full_head_ablation(model, loader, device, emotion_names, baseline_f1):
    """Ablate each of 144 heads and measure per-emotion F1 drop.
    
    Returns:
        importance_matrix: (144, 28) array of F1 drops (positive = head matters)
        results_df: DataFrame with all results
    """
    n_layers = 12
    n_heads = 12
    n_emotions = len(emotion_names)
    
    importance_matrix = np.zeros((n_layers * n_heads, n_emotions))
    macro_drops = []
    
    total = n_layers * n_heads
    pbar = tqdm(total=total, desc="Ablating heads")
    
    for layer_idx in range(n_layers):
        for head_idx in range(n_heads):
            flat_idx = layer_idx * n_heads + head_idx
            
            # Install ablation hook
            handle = ablate_head_via_attention_self(model, layer_idx, head_idx)
            
            # Evaluate
            ablated_f1, ablated_macro = evaluate_per_emotion(model, loader, device)
            
            # Remove hook
            handle.remove()
            
            # Compute importance = baseline - ablated (positive means head was helpful)
            f1_drop = baseline_f1 - ablated_f1
            importance_matrix[flat_idx] = f1_drop
            macro_drops.append(baseline_macro - ablated_macro)
            
            pbar.set_postfix({
                'layer': layer_idx, 
                'head': head_idx,
                'macro_drop': f"{macro_drops[-1]:.4f}"
            })
            pbar.update(1)
    
    pbar.close()
    
    # Build results DataFrame
    records = []
    for layer_idx in range(n_layers):
        for head_idx in range(n_heads):
            flat_idx = layer_idx * n_heads + head_idx
            record = {
                'layer': layer_idx,
                'head': head_idx,
                'flat_idx': flat_idx,
                'macro_f1_drop': macro_drops[flat_idx],
            }
            for e_idx, e_name in enumerate(emotion_names):
                record[f'f1_drop_{e_name}'] = importance_matrix[flat_idx, e_idx]
            records.append(record)
    
    results_df = pd.DataFrame(records)
    return importance_matrix, results_df

In [ ]:
importance_matrix, ablation_df = run_full_head_ablation(
    model, eval_loader, device, emotion_names, baseline_per_emotion
)

print(f"Importance matrix shape: {importance_matrix.shape}")
print(f"\nTop 10 most important heads (by macro F1 drop):")
top_heads = ablation_df.nlargest(10, 'macro_f1_drop')
for _, row in top_heads.iterrows():
    print(f"  Layer {int(row['layer'])}, Head {int(row['head'])}: macro drop = {row['macro_f1_drop']:.4f}")

## 3. Head Importance Heatmap

In [ ]:
# Reshape importance matrix to (12 layers, 12 heads, 28 emotions)
importance_3d = importance_matrix.reshape(NUM_LAYERS, NUM_HEADS, NUM_EMOTIONS)

# Mean importance across emotions for each head
mean_importance = importance_3d.mean(axis=2)  # (12, 12)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    mean_importance,
    annot=True, fmt='.3f',
    xticklabels=[f'H{i}' for i in range(NUM_HEADS)],
    yticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    cmap='RdYlBu_r',
    center=0,
    ax=ax
)
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Mean Head Importance (F1 Drop When Ablated)')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'head_importance_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-emotion importance: which heads matter most for each emotion?
fig, axes = plt.subplots(4, 7, figsize=(28, 16))

for e_idx in range(NUM_EMOTIONS):
    ax = axes[e_idx // 7, e_idx % 7]
    emotion_importance = importance_3d[:, :, e_idx]  # (12, 12)
    
    sns.heatmap(
        emotion_importance,
        cmap='RdYlBu_r', center=0,
        xticklabels=False, yticklabels=False,
        cbar=False, ax=ax
    )
    ax.set_title(emotion_names[e_idx], fontsize=9)

plt.suptitle('Head Importance per Emotion (F1 Drop)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'head_importance_per_emotion.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Head Specialization Analysis

**Selectivity**: How specific is each head to a subset of emotions?
- A head with high F1 drop for 1-2 emotions but near-zero for the rest is **specialized**
- A head with uniform F1 drop across many emotions is **general-purpose**

We measure selectivity using the Gini coefficient of the absolute importance vector.

In [ ]:
def gini_coefficient(values):
    """Compute Gini coefficient. 0 = perfectly uniform, 1 = perfectly concentrated."""
    values = np.abs(values)
    if values.sum() == 0:
        return 0.0
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals) / (n * np.sum(sorted_vals))) - (n + 1) / n


# Compute selectivity for each head
head_selectivity = []
head_total_importance = []
head_top_emotions = []

for flat_idx in range(NUM_LAYERS * NUM_HEADS):
    imp = importance_matrix[flat_idx]  # (28,)
    selectivity = gini_coefficient(imp)
    total_imp = np.abs(imp).sum()
    top_3 = np.argsort(imp)[::-1][:3]
    top_emotions = [(emotion_names[i], imp[i]) for i in top_3]
    
    head_selectivity.append(selectivity)
    head_total_importance.append(total_imp)
    head_top_emotions.append(top_emotions)

head_selectivity = np.array(head_selectivity)
head_total_importance = np.array(head_total_importance)

In [ ]:
# Scatter: selectivity vs total importance
fig, ax = plt.subplots(figsize=(10, 8))

colors = [plt.cm.viridis(l / 11) for l in range(NUM_LAYERS) for _ in range(NUM_HEADS)]
layers_flat = [l for l in range(NUM_LAYERS) for _ in range(NUM_HEADS)]

scatter = ax.scatter(
    head_total_importance, head_selectivity,
    c=layers_flat, cmap='viridis', alpha=0.7, s=60, edgecolors='white', linewidth=0.5
)

# Label the most interesting heads (high importance OR high selectivity)
for flat_idx in range(NUM_LAYERS * NUM_HEADS):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    if head_total_importance[flat_idx] > np.percentile(head_total_importance, 95) or \
       (head_selectivity[flat_idx] > np.percentile(head_selectivity, 90) and 
        head_total_importance[flat_idx] > np.percentile(head_total_importance, 50)):
        ax.annotate(
            f'L{layer}H{head}', 
            (head_total_importance[flat_idx], head_selectivity[flat_idx]),
            fontsize=7, ha='center', va='bottom'
        )

plt.colorbar(scatter, ax=ax, label='Layer')
ax.set_xlabel('Total Importance (sum of |F1 drops|)')
ax.set_ylabel('Selectivity (Gini coefficient)')
ax.set_title('Head Specialization: Importance vs Selectivity')
ax.axhline(y=np.median(head_selectivity), color='gray', linestyle='--', alpha=0.5, label='Median selectivity')
ax.axvline(x=np.median(head_total_importance), color='gray', linestyle=':', alpha=0.5, label='Median importance')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'head_specialization_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorize heads into quadrants
med_imp = np.median(head_total_importance)
med_sel = np.median(head_selectivity)

categories = {
    'Critical Specialists': [],   # high importance, high selectivity
    'Critical Generalists': [],   # high importance, low selectivity  
    'Minor Specialists': [],      # low importance, high selectivity
    'Dispensable': [],            # low importance, low selectivity
}

for flat_idx in range(NUM_LAYERS * NUM_HEADS):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    high_imp = head_total_importance[flat_idx] > med_imp
    high_sel = head_selectivity[flat_idx] > med_sel
    
    if high_imp and high_sel:
        cat = 'Critical Specialists'
    elif high_imp and not high_sel:
        cat = 'Critical Generalists'
    elif not high_imp and high_sel:
        cat = 'Minor Specialists'
    else:
        cat = 'Dispensable'
    
    categories[cat].append((layer, head, head_total_importance[flat_idx], head_selectivity[flat_idx]))

print("Head Categories:")
print("="*60)
for cat_name, heads in categories.items():
    print(f"\n{cat_name} ({len(heads)} heads):")
    # Show top 5 by importance
    sorted_heads = sorted(heads, key=lambda x: x[2], reverse=True)[:5]
    for layer, head, imp, sel in sorted_heads:
        top_em = head_top_emotions[layer * NUM_HEADS + head]
        top_str = ', '.join([f"{e}({v:.3f})" for e, v in top_em])
        print(f"  L{layer}H{head}: imp={imp:.3f}, sel={sel:.3f} | top: {top_str}")

## 5. Emotion-Head Affinity: Which Heads "Own" Which Emotions?

In [ ]:
# For each emotion, find the most critical heads
print("Critical Heads per Emotion (top 5 by F1 drop):")
print("="*70)

emotion_head_mapping = {}

for e_idx, e_name in enumerate(emotion_names):
    imp = importance_matrix[:, e_idx]  # (144,)
    top_5_idx = np.argsort(imp)[::-1][:5]
    
    critical = []
    print(f"\n{e_name} (baseline F1={baseline_per_emotion[e_idx]:.3f}):")
    for idx in top_5_idx:
        layer = idx // NUM_HEADS
        head = idx % NUM_HEADS
        drop = imp[idx]
        print(f"  L{layer}H{head}: F1 drop = {drop:+.4f}")
        critical.append((layer, head, drop))
    
    emotion_head_mapping[e_name] = critical

In [ ]:
# Clustered heatmap: emotions x heads (top heads only for readability)
# Select top 30 most important heads overall
top_n_heads = 30
overall_importance = np.abs(importance_matrix).sum(axis=1)
top_head_indices = np.argsort(overall_importance)[::-1][:top_n_heads]

# Create submatrix
sub_matrix = importance_matrix[top_head_indices].T  # (28 emotions, top_n heads)
head_labels = [f"L{i//NUM_HEADS}H{i%NUM_HEADS}" for i in top_head_indices]

fig, ax = plt.subplots(figsize=(16, 10))
g = sns.clustermap(
    pd.DataFrame(sub_matrix, index=emotion_names, columns=head_labels),
    cmap='RdYlBu_r', center=0,
    figsize=(16, 10),
    row_cluster=True, col_cluster=True,
    yticklabels=True, xticklabels=True,
    dendrogram_ratio=(0.1, 0.1),
    cbar_kws={'label': 'F1 Drop (positive = head important)'}
)
g.fig.suptitle(f'Emotion-Head Affinity (Top {top_n_heads} Heads)', y=1.02, fontsize=14)
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'emotion_head_clustermap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Head Redundancy Analysis

Two heads are **redundant** if ablating either one alone has little effect, but ablating both together causes a large F1 drop. We approximate this by computing correlation between importance profiles.

In [ ]:
# Correlation between heads based on their emotion-importance profiles
head_correlation = np.corrcoef(importance_matrix)  # (144, 144)

# Find the most redundant pairs (highest correlation)
redundant_pairs = []
for i in range(NUM_LAYERS * NUM_HEADS):
    for j in range(i + 1, NUM_LAYERS * NUM_HEADS):
        corr = head_correlation[i, j]
        if not np.isnan(corr):
            redundant_pairs.append((i, j, corr))

redundant_pairs.sort(key=lambda x: x[2], reverse=True)

print("Top 15 Most Redundant Head Pairs (highest correlation):")
print("="*60)
for i, j, corr in redundant_pairs[:15]:
    l1, h1 = i // NUM_HEADS, i % NUM_HEADS
    l2, h2 = j // NUM_HEADS, j % NUM_HEADS
    print(f"  L{l1}H{h1} <-> L{l2}H{h2}: r = {corr:.4f}")

In [ ]:
# Visualize head redundancy as a block-diagonal pattern
fig, ax = plt.subplots(figsize=(12, 10))

# Reorder by layer for visual clarity
layer_labels = [f'L{i//NUM_HEADS}H{i%NUM_HEADS}' for i in range(NUM_LAYERS * NUM_HEADS)]

# Show only inter-layer structure (average correlation between layers)
layer_corr = np.zeros((NUM_LAYERS, NUM_LAYERS))
for l1 in range(NUM_LAYERS):
    for l2 in range(NUM_LAYERS):
        block = head_correlation[
            l1*NUM_HEADS:(l1+1)*NUM_HEADS,
            l2*NUM_HEADS:(l2+1)*NUM_HEADS
        ]
        # Mean of off-diagonal elements if same layer, all elements if different
        if l1 == l2:
            mask = ~np.eye(NUM_HEADS, dtype=bool)
            layer_corr[l1, l2] = np.nanmean(block[mask])
        else:
            layer_corr[l1, l2] = np.nanmean(block)

sns.heatmap(
    layer_corr, annot=True, fmt='.2f',
    xticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    yticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    cmap='RdYlBu_r', center=0, vmin=-1, vmax=1,
    ax=ax
)
ax.set_title('Mean Head Importance Correlation Between Layers')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'head_redundancy_layers.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Attention Pattern Visualization

For the most specialized heads, visualize what they attend to on example sentences.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Select top 6 most specialized heads (high selectivity AND importance)
specialist_score = head_selectivity * head_total_importance
top_specialist_indices = np.argsort(specialist_score)[::-1][:6]

print("Top 6 Specialist Heads:")
for idx in top_specialist_indices:
    layer = idx // NUM_HEADS
    head = idx % NUM_HEADS
    top_em = head_top_emotions[idx]
    print(f"  L{layer}H{head}: selectivity={head_selectivity[idx]:.3f}, "
          f"importance={head_total_importance[idx]:.3f}")
    print(f"    Top emotions: {', '.join([f'{e}({v:.3f})' for e, v in top_em])}")

In [ ]:
# Example sentences for visualization
example_sentences = [
    "I am so happy and grateful for everything you've done!",
    "This is absolutely disgusting and makes me angry.",
    "I'm really scared about what might happen next.",
    "That's hilarious, I can't stop laughing!",
    "I feel so sad and lonely right now.",
    "Wow, I'm genuinely surprised by this news."
]

@torch.no_grad()
def get_attention_weights(model, text, tokenizer, device):
    """Get attention weights for all layers and heads."""
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    outputs = model.bert(**inputs, output_attentions=True)
    # attentions is a tuple of (n_layers,) each (batch, n_heads, seq_len, seq_len)
    attentions = outputs.attentions
    
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu())
    
    return attentions, tokens

In [ ]:
# Visualize attention patterns for top specialist heads on example sentences
n_specialist = min(4, len(top_specialist_indices))
n_sentences = min(3, len(example_sentences))

fig, axes = plt.subplots(n_specialist, n_sentences, figsize=(6*n_sentences, 5*n_specialist))
if n_specialist == 1:
    axes = axes.reshape(1, -1)
if n_sentences == 1:
    axes = axes.reshape(-1, 1)

for s_idx, head_flat_idx in enumerate(top_specialist_indices[:n_specialist]):
    layer = head_flat_idx // NUM_HEADS
    head = head_flat_idx % NUM_HEADS
    top_em = head_top_emotions[head_flat_idx]
    
    for sent_idx in range(n_sentences):
        attentions, tokens = get_attention_weights(
            model, example_sentences[sent_idx], tokenizer, device
        )
        
        attn = attentions[layer][0, head].cpu().numpy()  # (seq_len, seq_len)
        n_tokens = len(tokens)
        attn = attn[:n_tokens, :n_tokens]
        
        ax = axes[s_idx, sent_idx]
        sns.heatmap(
            attn, xticklabels=tokens, yticklabels=tokens,
            cmap='Blues', ax=ax, cbar=False,
            square=True
        )
        ax.set_title(f'L{layer}H{head} ({top_em[0][0]})', fontsize=9)
        ax.tick_params(axis='both', labelsize=6)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        plt.setp(ax.get_yticklabels(), rotation=0)

plt.suptitle('Attention Patterns of Top Specialist Heads', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'specialist_head_attention.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Layer-wise Head Importance Distribution

In [ ]:
# How is head importance distributed across layers?
layer_importance = []
for l in range(NUM_LAYERS):
    layer_heads = importance_matrix[l*NUM_HEADS:(l+1)*NUM_HEADS]
    layer_importance.append({
        'layer': l,
        'mean_abs_importance': np.abs(layer_heads).mean(),
        'max_abs_importance': np.abs(layer_heads).max(),
        'n_positive_heads': (layer_heads.mean(axis=1) > 0).sum(),
        'n_negative_heads': (layer_heads.mean(axis=1) < 0).sum(),
    })

layer_imp_df = pd.DataFrame(layer_importance)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of layer importance
axes[0].bar(layer_imp_df['layer'], layer_imp_df['mean_abs_importance'], color='steelblue')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Mean |F1 Drop|')
axes[0].set_title('Layer-wise Head Importance')
axes[0].set_xticks(range(NUM_LAYERS))

# Boxplot of per-head importance by layer
layer_data = []
for l in range(NUM_LAYERS):
    for h in range(NUM_HEADS):
        flat_idx = l * NUM_HEADS + h
        mean_drop = importance_matrix[flat_idx].mean()
        layer_data.append({'Layer': f'L{l}', 'Mean F1 Drop': mean_drop})

layer_box_df = pd.DataFrame(layer_data)
sns.boxplot(data=layer_box_df, x='Layer', y='Mean F1 Drop', ax=axes[1], color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Head Importance Distribution by Layer')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'layer_head_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Compression Implications Summary

In [ ]:
# Summary statistics for informed compression
print("="*70)
print("HEAD ANALYSIS SUMMARY — Implications for Informed Compression")
print("="*70)

# Dispensable heads = safe to compress aggressively
dispensable = categories['Dispensable']
critical_spec = categories['Critical Specialists']
critical_gen = categories['Critical Generalists']

print(f"\n1. DISPENSABLE HEADS ({len(dispensable)} heads):")
print(f"   These heads can be compressed more aggressively (lower SVD rank).")
disp_layers = [h[0] for h in dispensable]
from collections import Counter
print(f"   Layer distribution: {dict(sorted(Counter(disp_layers).items()))}")

print(f"\n2. CRITICAL SPECIALIST HEADS ({len(critical_spec)} heads):")
print(f"   These heads should be preserved (higher SVD rank).")
print(f"   Compressing these risks destroying specific emotions.")
for layer, head, imp, sel in sorted(critical_spec, key=lambda x: x[2], reverse=True)[:10]:
    top_em = head_top_emotions[layer * NUM_HEADS + head]
    print(f"   L{layer}H{head}: {', '.join([f'{e}' for e, v in top_em[:2]])}")

print(f"\n3. CRITICAL GENERALIST HEADS ({len(critical_gen)} heads):")
print(f"   These heads serve multiple emotions — moderate compression.")

# Per-layer compression recommendation
print(f"\n4. PER-LAYER ATTENTION COMPRESSION RECOMMENDATION:")
for l in range(NUM_LAYERS):
    layer_heads_imp = head_total_importance[l*NUM_HEADS:(l+1)*NUM_HEADS]
    mean_imp = layer_heads_imp.mean()
    max_imp = layer_heads_imp.max()
    
    if mean_imp > np.percentile(head_total_importance, 75):
        rec = "PRESERVE (high rank)"
    elif mean_imp > np.percentile(head_total_importance, 25):
        rec = "MODERATE compression"
    else:
        rec = "AGGRESSIVE compression (low rank)"
    
    print(f"   Layer {l:2d}: mean_imp={mean_imp:.4f}, max_imp={max_imp:.4f} -> {rec}")

In [ ]:
# Save results
results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)

# Save ablation results
ablation_df.to_csv(os.path.join(results_dir, 'head_ablation_results.csv'), index=False)

# Save importance matrix
np.save(os.path.join(results_dir, 'head_importance_matrix.npy'), importance_matrix)

# Save head categorization
cat_records = []
for cat_name, heads in categories.items():
    for layer, head, imp, sel in heads:
        cat_records.append({
            'layer': layer, 'head': head, 'category': cat_name,
            'total_importance': imp, 'selectivity': sel
        })
cat_df = pd.DataFrame(cat_records)
cat_df.to_csv(os.path.join(results_dir, 'head_categories.csv'), index=False)

# Save per-layer recommendation
layer_rec = []
for l in range(NUM_LAYERS):
    layer_heads_imp = head_total_importance[l*NUM_HEADS:(l+1)*NUM_HEADS]
    layer_rec.append({
        'layer': l,
        'mean_head_importance': layer_heads_imp.mean(),
        'max_head_importance': layer_heads_imp.max(),
        'mean_selectivity': head_selectivity[l*NUM_HEADS:(l+1)*NUM_HEADS].mean()
    })
pd.DataFrame(layer_rec).to_csv(os.path.join(results_dir, 'layer_attention_importance.csv'), index=False)

print("Results saved to results/")
print(f"  - head_ablation_results.csv ({len(ablation_df)} rows)")
print(f"  - head_importance_matrix.npy {importance_matrix.shape}")
print(f"  - head_categories.csv ({len(cat_df)} rows)")
print(f"  - layer_attention_importance.csv")